# Section 1 — Getting Started with Claude API
**Course:** Building with the Claude API — Anthropic Academy  
**Practice focus:** QA Manual Engineer learning AI automation  

---
This notebook follows Section 1 of the course. All examples use QA-related prompts so you can practice with real use cases.

**What you'll learn:**
1. Install the Anthropic SDK
2. Authenticate with your API key
3. Send your first message
4. Understand the response structure
5. Stream responses in real time
6. Build a multi-turn conversation


## 1. Install Required Libraries
Install the Anthropic Python SDK. Run this cell once.

In [ ]:
# Install the official Anthropic Python SDK
# python-dotenv is used to load your API key from the .env file safely
%pip install anthropic python-dotenv --quiet

## 2. Set Up API Authentication
Load your API key from the `.env` file and create the Anthropic client.  
> ⚠️ Never hardcode your API key directly in code — always use environment variables.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

# Load the .env file from the project root
# This reads ANTHROPIC_API_KEY=your_key_here from the .env file
load_dotenv(dotenv_path="../.env")

# Create the Anthropic client — this is the main object you use to talk to Claude
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("✅ Client created successfully!")

## 3. Send Your First Message
Use `client.messages.create()` to send a message to Claude.  
This is the core method you'll use for everything in this course.

In [ ]:
# Send a message to Claude — QA example: ask it to explain a testing concept
response = client.messages.create(
    model="claude-3-5-sonnet-latest",   # The Claude model to use
    max_tokens=300,                      # Maximum number of tokens in the response
    messages=[
        {
            "role": "user",              # "user" = message from you
            "content": "In 2 sentences, what is the difference between Smoke Testing and Regression Testing?"
        }
    ]
)

# Print just the text of Claude's reply
print(response.content[0].text)

## 4. Understand the Response Structure
The response object contains more than just the text. Let's explore what Claude returns.

In [ ]:
# Let's inspect the full response object
print("=== RESPONSE STRUCTURE ===")
print(f"Model used       : {response.model}")
print(f"Stop reason      : {response.stop_reason}")   # why Claude stopped — 'end_turn' = finished normally
print(f"Input tokens     : {response.usage.input_tokens}")   # tokens in your message (costs money)
print(f"Output tokens    : {response.usage.output_tokens}")  # tokens in Claude's reply (costs money)
print(f"Number of blocks : {len(response.content)}")         # usually 1 text block
print()
print("=== CONTENT BLOCK ===")
print(f"Type : {response.content[0].type}")   # 'text' for normal responses
print(f"Text : {response.content[0].text}")

## 5. Stream Responses
Streaming lets you see Claude's reply word-by-word as it's being generated — like ChatGPT's typing effect.  
Useful when generating long outputs like full test suites.

In [ ]:
# Stream Claude's response token by token
# end="" and flush=True make the text appear word-by-word without newlines between each token
print("Claude is generating test cases...\n")

with client.messages.stream(
    model="claude-3-5-sonnet-latest",
    max_tokens=400,
    messages=[
        {
            "role": "user",
            "content": "List 3 quick smoke test cases for a login feature. Format: TC-001, TC-002, TC-003."
        }
    ]
) as stream:
    # Each chunk is a small piece of the response as it arrives
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)

print("\n\n✅ Streaming complete!")

## 6. Multi-turn Conversation
Claude doesn't remember previous messages automatically — you have to send the full conversation history each time.  
This is how you build a back-and-forth dialogue (like a chat).

> **QA use case:** Ask Claude to generate test cases, then ask it to add more edge cases in the next turn.

In [ ]:
# conversation_history stores all messages sent and received
# Claude needs the full history to understand the context of each new message
conversation_history = []

def chat(user_message):
    """Send a message and get a reply. Keeps full conversation history."""
    
    # Add the user's new message to the history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    # Send the full history to Claude every time
    response = client.messages.create(
        model="claude-3-5-sonnet-latest",
        max_tokens=500,
        system="You are a senior QA engineer. Give concise, practical answers.",  # sets Claude's role
        messages=conversation_history
    )
    
    assistant_reply = response.content[0].text
    
    # Add Claude's reply to history so the next turn remembers it
    conversation_history.append({
        "role": "assistant",
        "content": assistant_reply
    })
    
    return assistant_reply


# --- Turn 1: Ask for test cases ---
print("👤 User: Give me 2 positive test cases for a login feature.\n")
reply1 = chat("Give me 2 positive test cases for a login feature.")
print(f"🤖 Claude:\n{reply1}\n")
print("-" * 60)

# --- Turn 2: Follow up — Claude remembers Turn 1 ---
print("👤 User: Now add 2 security test cases for the same feature.\n")
reply2 = chat("Now add 2 security test cases for the same feature.")
print(f"🤖 Claude:\n{reply2}\n")
print("-" * 60)

# --- Turn 3: Follow up again ---
print("👤 User: Which of these 4 test cases should be in the Smoke suite?\n")
reply3 = chat("Which of these 4 test cases should be in the Smoke suite?")
print(f"🤖 Claude:\n{reply3}")

---
## 7. [Vertex AI] Making a Request via Google Cloud
Bài học này dạy cách gọi Claude qua **Google Cloud Vertex AI** thay vì Anthropic API trực tiếp.  
Dùng khi công ty yêu cầu data nằm trong GCP, hoặc muốn dùng chung GCP billing.

> **Yêu cầu trước:** Đã cài gcloud CLI và chạy `gcloud auth application-default login`

In [ ]:
# Bước 1: Cài SDK với Vertex AI support
# [vertex] là extra package để kết nối Google Cloud
%pip install "anthropic[vertex]" --quiet


In [ ]:
from anthropic import AnthropicVertex

# Bước 2: Tạo client Vertex AI
# - region: nơi model chạy (us-east5 hỗ trợ Claude)
# - project_id: lấy từ Google Cloud Console
# Không cần API key — dùng gcloud credentials tự động
GCP_PROJECT_ID = "your-project-id"  # <-- thay bằng project ID thực của bạn

vertex_client = AnthropicVertex(
    region="us-east5",
    project_id=GCP_PROJECT_ID,
)

# Đặt model làm biến để tái sử dụng
model = "claude-sonnet-4@20250514"

print("✅ Vertex AI client created!")
print(f"   Project: {GCP_PROJECT_ID}")
print(f"   Region : us-east5")
print(f"   Model  : {model}")


In [ ]:
# Bước 3: Gọi Claude qua Vertex AI — cú pháp GIỐNG HỆT Anthropic API trực tiếp
# Chỉ khác: dùng vertex_client thay vì client
message = vertex_client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {
            "role": "user",   # "user" = tin nhắn từ bạn
            "content": "In 2 sentences, what is the difference between Smoke Testing and Regression Testing?"
        }
    ]
)

# Lấy text từ response — cú pháp giống hệt Anthropic API trực tiếp
print(message.content[0].text)


### So sánh: Anthropic API trực tiếp vs Vertex AI

| | Anthropic API | Vertex AI |
|---|---|---|
| **Import** | `from anthropic import Anthropic` | `from anthropic import AnthropicVertex` |
| **Auth** | `api_key=os.environ.get("ANTHROPIC_API_KEY")` | gcloud credentials tự động |
| **Client** | `Anthropic()` | `AnthropicVertex(region=..., project_id=...)` |
| **Model name** | `claude-3-5-sonnet-latest` | `claude-sonnet-4@20250514` |
| **Gọi API** | `client.messages.create(...)` | `vertex_client.messages.create(...)` ← **giống hệt** |
| **Đọc response** | `message.content[0].text` | `message.content[0].text` ← **giống hệt** |

> **Takeaway:** Chỉ cần đổi cách khởi tạo client. Toàn bộ code còn lại (messages, max_tokens, response) **không đổi**.